# 단지별 세대수 / 도시별 세대수 변수 추가
- 국토교통부와 건축물대장 CSV를 이용해 단지명을 매칭하여 세대수 변수값 수집, 단지별_세대수 값을 가지고 도시별_세대수 값 수집
- `단지별_세대수`, `도시별_세대수` 컬럼 추가
- 쓰여진 함수 설명

    tokenize()
    → 단지명을 토큰으로 쪼개는 함수

    token_similarity()
    → 두 단지명의 유사도 점수 계산
    → tokenize() 사용

    best_match_from()
    → df 전체에서 가장 유사한 단지명 찾기
    → token_similarity() 사용

    find_best_match()  ← 메인 매칭 함수
    → 위 함수들을 모두 조합해서 최종 세대수 반환
    → best_match_from() 사용

In [ ]:
import pandas as pd
import re

In [ ]:
# 데이터 불러오기 (원본 데이터의 단지명과 정확히 일치하는 단지명을 찾기 위해서 두개의 CSV 데이터 이용)

# 국토교통부_공동주택 단지 기본 정보 csv 불러오기
kapt_df = pd.read_csv("국토교통부_공동주택_단지기본정보.csv", header=1, skiprows=[2], encoding="utf-8-sig")

# 건축물대장 표제부 csv 불러오고 동별 세대수를 합쳐서 단지별 세대수로 만들기
건축물대장_df = pd.read_csv(r"건축물대장_표제부.csv", encoding="utf-8-sig", low_memory=False)
건축물_households = (
    건축물대장_df[건축물대장_df["세대수(세대)"] > 0]
    .groupby("건물명", as_index=False)["세대수(세대)"]
    .sum()
    .rename(columns={"건물명": "단지명", "세대수(세대)": "세대수"})
)

# 변수 추가할 CSV 데이터 불러오기
df = pd.read_csv("new_city.csv", encoding="utf-8-sig")
print("데이터 크기:", df.shape)
df.head()

In [ ]:
# 숫자 추출 함수 (다른 단지끼리 매칭되는걸 막기 위함)
def extract_numbers(name):
    return set(re.findall(r'\d+', str(name)))

In [ ]:
# 토큰화 하는 함수
def tokenize(name):
    name = re.sub(r'[()（）]', ' ', str(name))
    tokens = re.findall(r'[가-힣]+|\d+|[a-zA-Z]+', name)
    return set(t.lower() for t in tokens)

# 단지명을 토큰으로 쪼개서 공통 토큰 비율로 유사도를 계산하는 함수
def token_similarity(name1, name2):
    t1 = tokenize(name1)
    t2 = tokenize(name2)

    nums1 = set(t for t in t1 if t.isdigit())
    nums2 = set(t for t in t2 if t.isdigit())
    if nums1 and nums2 and nums1 != nums2:
        return 0.0

    intersection = t1 & t2
    union = t1 | t2
    if not union:
        return 0.0
    return len(intersection) / len(union)

In [ ]:
# "경기도 성남시 분당구 백현동" → "분당구", "백현동" 로 추출하는 함수들

# 시군구 키 추출 함수
def get_sigungu_key(sigungu_str):
    parts = str(sigungu_str).split()
    result = ""
    for p in parts:
        if p.endswith(("시", "군", "구")):
            result = p
    return result

# 동리 키 추출 함수
def get_dong_key(sigungu_str):
    parts = str(sigungu_str).split()
    for p in parts:
        if p.endswith(("동", "리", "읍", "면")):
            return p
    return ""

In [ ]:
# 단지명 괄호 안 숫자 직접 추출 함수
# 예) 해솔마을1단지(647) → 647

def extract_households_from_name(apt_name):
    match = re.search(r'\((\d+)\)$', str(apt_name))
    return int(match.group(1)) if match else None

In [ ]:
# Source_df에서 토큰 유사도 매칭 함수
def best_match_from(apt_name, source_df, cutoff=0.4):
    best_name, best_score = None, 0.0
    for kapt_name in source_df["단지명"].astype(str):
        score = token_similarity(apt_name, kapt_name)
        if score > best_score:
            best_score = score
            best_name = kapt_name
    if best_score >= cutoff:
        households = source_df[source_df["단지명"] == best_name]["세대수"].values[0]
        return best_name, households, best_score
    return None, None, 0.0

In [ ]:
# 메인 매칭 함수
# 0순위: 세대수 적혀있는 괄호 안 숫자 직접 추출
# 1순위: kapt_df (시군구+동 필터 후 유사도 매칭)
# 2순위: 건축물대장 (유사도 매칭)

def find_best_match(apt_name, sigungu, kapt_df, 건축물_df, cutoff=0.4):
    # 0순위: 단지명 괄호 안 숫자 직접 추출
    direct = extract_households_from_name(apt_name)
    if direct:
        return apt_name, direct, "괄호추출"

    sigungu_key = get_sigungu_key(sigungu)
    dong_key = get_dong_key(sigungu)

    def apply_filter(source_df):
        filtered = source_df[
            source_df["시군구"].astype(str).str.replace(" ", "").str.contains(
                sigungu_key, na=False
            )
        ] if "시군구" in source_df.columns else source_df

        if dong_key and not filtered.empty:
            col = "법정동주소" if "법정동주소" in source_df.columns else "단지명"
            filtered_dong = filtered[filtered[col].astype(str).str.contains(dong_key, na=False)]
            if not filtered_dong.empty:
                filtered = filtered_dong
        return filtered if not filtered.empty else source_df

    # 1순위: kapt_df
    filtered_kapt = apply_filter(kapt_df)
    name1, h1, s1 = best_match_from(apt_name, filtered_kapt, cutoff)
    if name1:
        return name1, h1, "kapt"

    # 2순위: 건축물대장
    name2, h2, s2 = best_match_from(apt_name, 건축물_households, cutoff)
    if name2:
        return name2, h2, "건축물대장"

    return None, None, "실패"

In [ ]:
# 매칭 준비
# 중복 제거한 고유 단지명 + 시군구 목록 추출
unique_combos = df[["단지명", "시군구"]].drop_duplicates()
apt_to_households = {}
unmatched = []

In [ ]:
# 매칭 실행
for _, row in unique_combos.iterrows():
    apt_name = row["단지명"]
    sigungu  = row["시군구"]
    matched_name, households, source = find_best_match(apt_name, sigungu, kapt_df, 건축물_households)

    if matched_name:
        apt_to_households[apt_name] = households
        print(f"  ✔ [{source}] {apt_name} → {matched_name} ({int(households)}세대)")
    else:
        apt_to_households[apt_name] = None
        unmatched.append(apt_name)
        print(f"  ✘ [실패] {apt_name}")

In [ ]:
# 단지별_세대수 컬럼 추가 및 저장
df["단지별_세대수"] = df["단지명"].apply(lambda x: apt_to_households[x])

print(f"\n매칭 실패 단지: {unmatched}")
print(f"세대수 조회 실패 건수: {df['단지별_세대수'].isna().sum()}건")
print(df["단지별_세대수"].describe())

# 매칭 실패 행 제거
df = df.dropna(subset=["단지별_세대수"])
print(f"실패 행 제거 후 남은 행 수: {len(df)}행")

df.to_csv("new_city.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료")